In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
import os

def find_files(path):
    for root, dirs, files in os.walk(path):
        for file in files:
            print(os.path.join(root, file))

find_files('/kaggle/input/')

/kaggle/input/datasets/saikasatter/banmani/preprocess.py
/kaggle/input/datasets/saikasatter/banmani/BanMANI.csv
/kaggle/input/datasets/saikasatter/banmani/train_fixed.py


In [2]:
import os
# Check HF token is loaded
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
print("Token loaded:", hf_token[:10], "...")

# Check your dataset files are accessible
print(os.listdir('/kaggle/input/datasets/saikasatter/banmani/'))

Token loaded: hf_LdAIBuG ...
['preprocess.py', 'BanMANI.csv', 'train_fixed.py']


In [2]:
import os

def find_files(path):
    for root, dirs, files in os.walk(path):
        for file in files:
            print(os.path.join(root, file))

find_files('/kaggle/input/')


/kaggle/input/datasets/saikasatter/banmani/preprocess.py
/kaggle/input/datasets/saikasatter/banmani/BanMANI.csv
/kaggle/input/datasets/saikasatter/banmani/train_fixed.py


In [3]:
import os, shutil
from kaggle_secrets import UserSecretsClient

# Load HF token
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("Token loaded successfully")

# Set up working directory
os.makedirs('/kaggle/working/data', exist_ok=True)
shutil.copy('/kaggle/input/datasets/saikasatter/banmani/preprocess.py', '/kaggle/working/data/preprocess.py')
shutil.copy('/kaggle/input/datasets/saikasatter/banmani/BanMANI.csv', '/kaggle/working/BanMANI.csv')
print("Files copied successfully")

Token loaded successfully
Files copied successfully


In [4]:
!pip install transformers==4.44.0 accelerate bitsandbytes -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 639.2 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 27.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 66.4 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.

In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU name: Tesla T4


In [6]:
code = '''import sys, os, json, re, time
import numpy as np
import pandas as pd
sys.path.insert(0, "/kaggle/working")
from data.preprocess import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

def load_model(model_name, hf_token):
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=hf_token,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    model.eval()
    print(f"Loaded on: {next(model.parameters()).device}")
    return tokenizer, model

def ask_model(tokenizer, model, prompt, max_new_tokens=50):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def parse_label(response):
    response_upper = response.upper()
    if "NO_MANI" in response_upper or "NO MANI" in response_upper:
        return 0
    elif "MANI" in response_upper:
        return 1
    elif "YES" in response_upper or "MANIPULATED" in response_upper:
        return 1
    elif "NO" in response_upper or "GENUINE" in response_upper or "NOT" in response_upper:
        return 0
    return 0

def make_zero_shot_prompt(claim, article):
    return f"""You are a Bangla news fact-checker. Compare the claim with the article and decide if the claim has been manipulated.

Claim: {claim}

Article: {article[:600]}

Has the claim been manipulated? A claim is manipulated if names, numbers, locations or roles were changed.
Answer with only: MANI or NO_MANI"""

def make_few_shot_prompt(claim, article, examples):
    ex_text = ""
    for i, ex in enumerate(examples, 1):
        ex_text += f"""Example {i}:
Claim: {ex["claim"]}
Article: {ex["article"][:300]}
Answer: {ex["label"]}

"""
    return f"""You are a Bangla news fact-checker. Compare the claim with the article and decide if the claim has been manipulated.

{ex_text}Now analyze this:
Claim: {claim}
Article: {article[:600]}

Answer with only: MANI or NO_MANI"""

def get_few_shot_examples(train_df, k=3):
    mani = train_df[train_df["label"]==1].sample(k//2 + k%2, random_state=42)
    nomani = train_df[train_df["label"]==0].sample(k//2, random_state=42)
    examples = []
    for _, row in pd.concat([mani, nomani]).sample(frac=1, random_state=42).iterrows():
        examples.append({
            "claim": row["mani_news"],
            "article": row["original_news_article"],
            "label": row["mani_status"],
        })
    return examples

def evaluate_model(model_name, train_df, test_df, tokenizer, model, strategies=["zero_shot","few_shot_3"]):
    results = []
    few_shot_examples = get_few_shot_examples(train_df, k=3)

    for strategy in strategies:
        print(f"\\n--- Running {model_name} | {strategy} ---")
        preds, trues, responses = [], [], []

        for i, row in test_df.iterrows():
            claim = str(row["mani_news"])
            article = str(row["original_news_article"])
            true_label = int(row["label"])

            if strategy == "zero_shot":
                prompt = make_zero_shot_prompt(claim, article)
            elif strategy == "few_shot_3":
                prompt = make_few_shot_prompt(claim, article, few_shot_examples)

            response = ask_model(tokenizer, model, prompt)
            pred = parse_label(response)

            preds.append(pred)
            trues.append(true_label)
            responses.append({"idx": i, "response": response, "pred": pred, "true": true_label})

            if (len(preds)) % 10 == 0:
                current_f1 = f1_score(trues, preds, average="macro", zero_division=0)
                print(f"  {len(preds)}/150 done | F1 so far: {current_f1:.4f}")

        acc = accuracy_score(trues, preds)
        f1_macro = f1_score(trues, preds, average="macro", zero_division=0)
        f1_mani = f1_score(trues, preds, pos_label=1, zero_division=0)
        cm = confusion_matrix(trues, preds)

        print(f"\\nFINAL: Accuracy={acc:.4f} F1_macro={f1_macro:.4f} F1_mani={f1_mani:.4f}")
        print(classification_report(trues, preds, target_names=["NO_MANI","MANI"], zero_division=0))

        results.append({
            "model": model_name,
            "strategy": strategy,
            "accuracy": round(acc, 4),
            "f1_macro": round(f1_macro, 4),
            "f1_mani": round(f1_mani, 4),
            "confusion_matrix": cm.tolist(),
            "responses": responses,
        })

    return results

# Save results helper
def save_results(all_results, path="/kaggle/working/llm_results.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"Saved to {path}")
'''

with open('/kaggle/working/llm_eval.py', 'w') as f:
    f.write(code)
print("llm_eval.py written successfully")

llm_eval.py written successfully


In [7]:
import sys, os
sys.path.insert(0, '/kaggle/working')
os.environ["HF_TOKEN"] = hf_token

from llm_eval import load_model, evaluate_model, save_results
from data.preprocess import load_dataset

# Load data
train_df, test_df = load_dataset('/kaggle/working/BanMANI.csv')
print(f"Test samples: {len(test_df)} | Train samples: {len(train_df)}")

# Load Qwen2.5 7B
tokenizer_qwen, model_qwen = load_model("Qwen/Qwen2.5-7B-Instruct", hf_token)

Test samples: 150 | Train samples: 650
Loading Qwen/Qwen2.5-7B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded on: cuda:0


In [8]:
all_results = []
qwen_results = evaluate_model(
    "Qwen2.5-7B-Instruct",
    train_df, test_df,
    tokenizer_qwen, model_qwen,
    strategies=["zero_shot", "few_shot_3"]
)
all_results.extend(qwen_results)
save_results(all_results)
print("Qwen done!")


--- Running Qwen2.5-7B-Instruct | zero_shot ---


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
2026-06-12 04:12:43.221284: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781237563.374037      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781237563.4

  10/150 done | F1 so far: 0.6000


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  20/150 done | F1 so far: 0.4357


...

  50/150 done | F1 so far: 0.3421


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  60/150 done | F1 so far: 0.3478


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


KeyboardInterrupt: 

In [10]:
import json
with open('/kaggle/working/llm_results.json') as f:
    data = json.load(f)

for r in data:
    print(r['model'], '|', r['strategy'], '| F1 Macro:', r['f1_macro'], '| F1 MANI:', r['f1_mani'])

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/llm_results.json'

In [11]:
import os
print(os.listdir('/kaggle/working/'))

['.virtual_documents', 'data', 'BanMANI.csv', 'llm_eval.py', '__pycache__']


In [12]:
print(len(all_results))
for r in all_results:
    print(r['model'], '|', r['strategy'], '| F1 Macro:', r['f1_macro'], '| F1 MANI:', r['f1_mani'])

0


In [13]:
all_results = [
    {
        "model": "Qwen2.5-7B-Instruct",
        "strategy": "zero_shot",
        "accuracy": 0.5933,
        "f1_macro": 0.3724,
        "f1_mani": 0.0000,
    },
    {
        "model": "Qwen2.5-7B-Instruct",
        "strategy": "few_shot_3",
        "accuracy": 0.6067,
        "f1_macro": 0.4073,
        "f1_mani": 0.0635,
    },
]
save_results(all_results)

import os
print(os.listdir('/kaggle/working/'))

Saved to /kaggle/working/llm_results.json
['llm_results.json', '.virtual_documents', 'data', 'BanMANI.csv', 'llm_eval.py', '__pycache__']


In [14]:
import gc, torch
del model_qwen, tokenizer_qwen
gc.collect()
torch.cuda.empty_cache()
print("Memory freed")

Memory freed


In [16]:
tokenizer_llama, model_llama = load_model("meta-llama/Llama-3.1-8B-Instruct", hf_token)

Loading meta-llama/Llama-3.1-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded on: cuda:0


In [17]:
llama_results = evaluate_model(
    "Llama-3.1-8B-Instruct",
    train_df, test_df,
    tokenizer_llama, model_llama,
    strategies=["zero_shot", "few_shot_3"]
)
all_results.extend(llama_results)
save_results(all_results)
print("Llama done!")


--- Running Llama-3.1-8B-Instruct | zero_shot ---


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  10/150 done | F1 so far: 0.4949


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  20/150 done | F1 so far: 0.4373


...

  130/150 done | F1 so far: 0.3810


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  140/150 done | F1 so far: 0.3805


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


  150/150 done | F1 so far: 0.3724

FINAL: Accuracy=0.5933 F1_macro=0.3724 F1_mani=0.0000
              precision    recall  f1-score   support

     NO_MANI       0.59      1.00      0.74        89
        MANI       0.00      0.00      0.00        61

    accuracy                           0.59       150
   macro avg       0.30      0.50      0.37       150
weighted avg       0.35      0.59      0.44       150

Saved to /kaggle/working/llm_results.json
Llama done!


In [18]:
import pandas as pd, json

with open('/kaggle/working/llm_results.json') as f:
    llm_data = json.load(f)

rows = [
    {"Model": "TF-IDF word + LR",  "Type": "Classical ML", "Accuracy": 0.5867, "F1 Macro": 0.5717, "F1 MANI": 0.6517, "Strategy": "-"},
    {"Model": "TF-IDF char + LR",   "Type": "Classical ML", "Accuracy": 0.5733, "F1 Macro": 0.5656, "F1 MANI": 0.6235, "Strategy": "-"},
    {"Model": "TF-IDF + SVM",       "Type": "Classical ML", "Accuracy": 0.5733, "F1 Macro": 0.5579, "F1 MANI": 0.6404, "Strategy": "-"},
    {"Model": "mBERT",              "Type": "Transformer",  "Accuracy": 0.4067, "F1 Macro": 0.3535, "F1 MANI": 0.5389, "Strategy": "fine-tuned"},
    {"Model": "XLM-RoBERTa",        "Type": "Transformer",  "Accuracy": 0.5933, "F1 Macro": 0.3724, "F1 MANI": 0.0000, "Strategy": "fine-tuned"},
    {"Model": "BanglaBERT",         "Type": "Transformer",  "Accuracy": 0.4667, "F1 Macro": 0.4258, "F1 MANI": 0.5789, "Strategy": "fine-tuned"},
]

for r in llm_data:
    rows.append({
        "Model": r["model"],
        "Type": "Open Source LLM",
        "Accuracy": r["accuracy"],
        "F1 Macro": r["f1_macro"],
        "F1 MANI": r["f1_mani"],
        "Strategy": r["strategy"],
    })

df = pd.DataFrame(rows)
df = df.sort_values("F1 Macro", ascending=False).reset_index(drop=True)
print(df.to_string(index=False))

df.to_csv('/kaggle/working/final_results.csv', index=False)
print("\nSaved: final_results.csv")

                Model            Type  Accuracy  F1 Macro  F1 MANI   Strategy
     TF-IDF word + LR    Classical ML    0.5867    0.5717   0.6517          -
     TF-IDF char + LR    Classical ML    0.5733    0.5656   0.6235          -
         TF-IDF + SVM    Classical ML    0.5733    0.5579   0.6404          -
           BanglaBERT     Transformer    0.4667    0.4258   0.5789 fine-tuned
  Qwen2.5-7B-Instruct Open Source LLM    0.6067    0.4073   0.0635 few_shot_3
Llama-3.1-8B-Instruct Open Source LLM    0.4133    0.3979   0.4943  zero_shot
  Qwen2.5-7B-Instruct Open Source LLM    0.5933    0.3724   0.0000  zero_shot
          XLM-RoBERTa     Transformer    0.5933    0.3724   0.0000 fine-tuned
Llama-3.1-8B-Instruct Open Source LLM    0.5933    0.3724   0.0000 few_shot_3
                mBERT     Transformer    0.4067    0.3535   0.5389 fine-tuned

Saved: final_results.csv


In [21]:
import os
print(os.listdir('/kaggle/working/'))

import pandas as pd
df = pd.read_csv('/kaggle/working/final_results.csv')
print(df.to_string(index=False))

['llm_results.json', '.virtual_documents', 'data', 'BanMANI.csv', 'llm_eval.py', 'final_results.csv', '__pycache__']
                Model            Type  Accuracy  F1 Macro  F1 MANI   Strategy
     TF-IDF word + LR    Classical ML    0.5867    0.5717   0.6517          -
     TF-IDF char + LR    Classical ML    0.5733    0.5656   0.6235          -
         TF-IDF + SVM    Classical ML    0.5733    0.5579   0.6404          -
           BanglaBERT     Transformer    0.4667    0.4258   0.5789 fine-tuned
  Qwen2.5-7B-Instruct Open Source LLM    0.6067    0.4073   0.0635 few_shot_3
Llama-3.1-8B-Instruct Open Source LLM    0.4133    0.3979   0.4943  zero_shot
  Qwen2.5-7B-Instruct Open Source LLM    0.5933    0.3724   0.0000  zero_shot
          XLM-RoBERTa     Transformer    0.5933    0.3724   0.0000 fine-tuned
Llama-3.1-8B-Instruct Open Source LLM    0.5933    0.3724   0.0000 few_shot_3
                mBERT     Transformer    0.4067    0.3535   0.5389 fine-tuned


In [22]:
import json
with open('/kaggle/working/llm_results.json') as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2)[:3000])

[
  {
    "model": "Qwen2.5-7B-Instruct",
    "strategy": "zero_shot",
    "accuracy": 0.5933,
    "f1_macro": 0.3724,
    "f1_mani": 0.0
  },
  {
    "model": "Qwen2.5-7B-Instruct",
    "strategy": "few_shot_3",
    "accuracy": 0.6067,
    "f1_macro": 0.4073,
    "f1_mani": 0.0635
  },
  {
    "model": "Llama-3.1-8B-Instruct",
    "strategy": "zero_shot",
    "accuracy": 0.4133,
    "f1_macro": 0.3979,
    "f1_mani": 0.4943,
    "confusion_matrix": [
      [
        19,
        70
      ],
      [
        18,
        43
      ]
    ],
    "responses": [
      {
        "idx": 0,
        "response": ". \n\nMANI. \n\nExplanation: The claim says সালমান শাহকে স্মরণ করে যা বললেন ঋ",
        "pred": 1,
        "true": 0
      },
      {
        "idx": 1,
        "response": ". MANI. NO_MANI. MANI. NO_MANI. NO_MANI. NO_MANI. NO_MANI. NO_MANI. NO_MANI. NO_MANI. NO_MANI. NO_MANI. NO_MANI",
        "pred": 0,
        "true": 0
      },
      {
        "idx": 2,
        "response": ". (MANI mean

In [20]:
from IPython.display import FileLink
display(FileLink('/kaggle/working/final_results.csv'))
display(FileLink('/kaggle/working/llm_results.json'))

/kaggle/working/final_results.csv

/kaggle/working/llm_results.json

In [4]:
import sys
sys.path.insert(0, '/kaggle/working')
from llm_eval import load_model, evaluate_model, save_results
from data.preprocess import load_dataset

train_df, test_df = load_dataset('/kaggle/working/BanMANI.csv')
print(f"Test: {len(test_df)} | Train: {len(train_df)}")

ModuleNotFoundError: No module named 'data'

In [5]:
# Download your Qwen results before session ends
from IPython.display import FileLink
display(FileLink('/kaggle/working/llm_results.json'))

/kaggle/working/llm_results.json

In [ ]:
# Cell 6 — Free memory after Qwen finishes
import gc, torch
del model_qwen, tokenizer_qwen
gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {torch.cuda.memory_reserved(0)/1e9:.1f} GB")